# TCG market parent brief — reproducible analysis

This notebook validates the purposive community sample and calculates the fee-friction scenarios used in the report. Counts describe the selected posts only; they are not population sentiment estimates.

In [1]:
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
if not (HERE / 'community-sample.csv').exists():
    HERE = HERE / 'output' / 'tcg-market-parent-brief-2026-07-15'
community = pd.read_csv(HERE / 'community-sample.csv')
theme_columns = [
    'nostalgia_ip', 'play_community', 'speculation_fomo', 'scarcity_scalping',
    'supply_reprint', 'affordability_macro', 'grading_financialization', 'exit_liquidity'
]
assert len(community) == 18
assert set(community[theme_columns].stack().unique()) <= {0, 1}
community[['region', 'platform', 'stance']].value_counts().head(10)

region  platform     stance  
US      Reddit       mixed       4
                     negative    3
        X            negative    3
Japan   5ch mirror   mixed       1
        Pokemon BBS  negative    1
        Talk.jp      negative    1
        X            negative    1
Taiwan  Disp.cc      negative    1
        PTT          mixed       1
                     positive    1
Name: count, dtype: int64

In [2]:
labels = {
    'nostalgia_ip': 'Nostalgia / IP affinity',
    'play_community': 'Play / community utility',
    'speculation_fomo': 'Speculation / FOMO',
    'scarcity_scalping': 'Scarcity / scalping',
    'supply_reprint': 'Supply / reprint response',
    'affordability_macro': 'Affordability / macro pressure',
    'grading_financialization': 'Grading / financialization',
    'exit_liquidity': 'Exit-liquidity concern',
}
theme_counts = (
    community[theme_columns].sum()
    .rename_axis('theme')
    .reset_index(name='posts')
)
theme_counts['label'] = theme_counts['theme'].map(labels)
theme_counts['share_of_sample'] = theme_counts['posts'] / len(community)
theme_counts = theme_counts.sort_values(['posts', 'label'], ascending=[False, True])
theme_counts[['label', 'posts', 'share_of_sample']]

                            label  posts  share_of_sample
5  Affordability / macro pressure     13         0.722222
2              Speculation / FOMO     13         0.722222
3             Scarcity / scalping      9         0.500000
7          Exit-liquidity concern      6         0.333333
4       Supply / reprint response      5         0.277778
0         Nostalgia / IP affinity      4         0.222222
1        Play / community utility      2         0.111111
6      Grading / financialization      1         0.055556

In [3]:
region_counts = community.groupby('region', as_index=False).size().rename(columns={'size': 'posts'})
stance_counts = community.groupby('stance', as_index=False).size().rename(columns={'size': 'posts'})
region_counts, stance_counts

(   region  posts
0   Japan      4
1  Taiwan      3
2      US     11,      stance  posts
0     mixed      6
1  negative     10
2  positive      2)

In [4]:
# Illustrative U.S. resale-friction scenario using a 13.25% final-value fee
# plus a $0.40 per-order fee. It excludes taxes, grading, returns, and acquisition friction.
purchase_price = 100.0
fee_rate = 0.1325
order_fee = 0.40
shipping_scenarios = [0.0, 5.0, 10.0]
break_even = []
for shipping in shipping_scenarios:
    required_sale = (purchase_price + order_fee + shipping) / (1 - fee_rate)
    break_even.append({
        'seller_paid_shipping': shipping,
        'required_sale_price': round(required_sale, 2),
        'required_appreciation_pct': round((required_sale / purchase_price - 1) * 100, 1),
    })
break_even = pd.DataFrame(break_even)
break_even

   seller_paid_shipping  required_sale_price  required_appreciation_pct
0                   0.0               115.73                       15.7
1                   5.0               121.50                       21.5
2                  10.0               127.26                       27.3

In [5]:
theme_counts.to_csv(HERE / 'derived-theme-counts.csv', index=False)
region_counts.to_csv(HERE / 'derived-region-counts.csv', index=False)
stance_counts.to_csv(HERE / 'derived-stance-counts.csv', index=False)
break_even.to_csv(HERE / 'derived-break-even.csv', index=False)
print('Validated 18 community discussions and wrote four derived datasets.')

Validated 18 community discussions and wrote four derived datasets.
